In [ ]:
from pathlib import Path

# Run this baseline for the attached XGLUE Kaggle dataset.
# Add the XGLUE dataset to the Kaggle notebook inputs before running all cells.
DATASET_KEYS = ("xglue4",)
RUN_LABEL = "fa_ast_gmn_baseline"


# XGLUE FA-AST+GMN Baseline

Pair-aware graph matching baseline for XGLUE. It uses the exported CPG graph as a FA-AST proxy and adds cross-graph node attention during message passing.


In [ ]:
# Executes the unchanged baseline pipeline once per dataset in isolated state.
# This run writes XGLUE-only CSV files, for example xglue4_*_results.csv.
def run_one_dataset(dataset_key: str):
    from pathlib import Path

    DATASET_KEY = dataset_key
    KAGGLE_DATA_ROOT = Path(f"/kaggle/input/datasets/koushamoeini/{DATASET_KEY}")
    WORK_DIR = Path("/kaggle/working")
    SEED = 42
    DEVICE = "cuda"

    METHOD_NAME = "FA-AST+GMN"
    GRAPH_TYPE = "cpg"

    MAX_TRAIN_PAIRS = None
    MAX_VALID_PAIRS = None
    MAX_TEST_PAIRS = None

    MAX_NODES = 96
    NODE_DIM = 4
    EPOCHS = 25
    BATCH_SIZE = 64
    HIDDEN_DIM = 192
    CODE_DIM = 128
    DROPOUT = 0.10
    LEARNING_RATE = 7e-4
    WEIGHT_DECAY = 1e-4
    PATIENCE = 6
    MESSAGE_STEPS = 4
    USE_AMP = True
    NUM_WORKERS = 0

    RESULTS_PATH = WORK_DIR / f"{DATASET_KEY}_fa_ast_gmn_baseline_results.csv"
    HISTORY_PATH = WORK_DIR / f"{DATASET_KEY}_fa_ast_gmn_baseline_training_history.csv"

    from pathlib import Path
    import gc
    import gzip
    import json
    import math
    import os
    import random
    import re
    import zipfile

    import numpy as np
    import pandas as pd
    from sklearn.metrics import accuracy_score, precision_recall_fscore_support
    from tqdm.auto import tqdm


    def seed_everything(seed: int = 42):
        random.seed(seed)
        np.random.seed(seed)
        try:
            import torch
            torch.manual_seed(seed)
            if torch.cuda.is_available():
                torch.cuda.manual_seed_all(seed)
        except Exception:
            pass


    def is_gzip_file(path: Path) -> bool:
        try:
            with path.open("rb") as f:
                return f.read(2) == b"\x1f\x8b"
        except OSError:
            return False


    def open_text(path: Path):
        return gzip.open(path, "rt", encoding="utf-8") if is_gzip_file(path) else path.open("r", encoding="utf-8")


    def candidate_roots():
        roots = [KAGGLE_DATA_ROOT, Path("/kaggle/input")]
        return [r for r in roots if r.exists()]


    def resolve_file_path(path: Path, *names: str) -> Path:
        if path.is_file():
            return path
        if path.is_dir():
            direct = [path / name for name in names if (path / name).is_file()]
            if direct:
                return direct[0]
            matches = []
            for name in names:
                matches.extend(path.rglob(name))
            matches = [m for m in matches if m.is_file()]
            if matches:
                return sorted(matches, key=lambda p: (len(p.relative_to(path).parts), len(str(p))))[0]
        return path


    def find_file(*names: str) -> Path:
        matches = []
        for root in candidate_roots():
            for name in names:
                direct = root / name
                if direct.is_file():
                    matches.append(direct)
                elif direct.is_dir():
                    resolved = resolve_file_path(direct, *names)
                    if resolved.is_file():
                        matches.append(resolved)
                matches.extend([p for p in root.rglob(name) if p.is_file()])
        if not matches:
            seen = []
            for root in candidate_roots():
                seen.extend(str(p) for p in sorted(root.rglob("*"))[:30])
            raise FileNotFoundError(f"Could not find {names}. First available paths: {seen}")
        return sorted(matches, key=lambda p: (len(p.parts), len(p.name), str(p)))[0]


    def load_pairs(path: Path) -> pd.DataFrame:
        path = resolve_file_path(path, "pairs.csv.gz", "pairs.csv", "pairs.csv.gz.tmp")
        compression = "gzip" if is_gzip_file(path) else None
        df = pd.read_csv(path, compression=compression, dtype={"left_id": str, "right_id": str, "split": str, "label": np.int64})
        df["left_id"] = df["left_id"].astype(str)
        df["right_id"] = df["right_id"].astype(str)
        return df


    def load_codes(path: Path) -> dict[str, str]:
        path = resolve_file_path(path, "codes.jsonl.gz", "codes.jsonl", "codes.jsonl.gz.tmp")
        codes = {}
        with open_text(path) as f:
            for line in tqdm(f, desc="Loading codes", unit="code"):
                if not line.strip():
                    continue
                obj = json.loads(line)
                codes[str(obj["code_id"])] = obj["code"]
        return codes


    def maybe_limit_split(df: pd.DataFrame, split: str, max_rows: int | None, seed: int) -> pd.DataFrame:
        part = df[df["split"] == split].copy()
        if max_rows is not None and len(part) > max_rows:
            part = part.sample(n=max_rows, random_state=seed)
        return part.reset_index(drop=True)


    def metric_dict(labels, scores, threshold: float) -> dict:
        pred = (np.asarray(scores) >= threshold).astype(np.int64)
        labels = np.asarray(labels).astype(np.int64)
        p, r, f1, _ = precision_recall_fscore_support(labels, pred, average="binary", zero_division=0)
        acc = accuracy_score(labels, pred)
        tp = int(((pred == 1) & (labels == 1)).sum())
        fp = int(((pred == 1) & (labels == 0)).sum())
        tn = int(((pred == 0) & (labels == 0)).sum())
        fn = int(((pred == 0) & (labels == 1)).sum())
        return {"P": p, "R": r, "F1": f1, "Acc": acc, "TP": tp, "FP": fp, "TN": tn, "FN": fn}


    def choose_threshold(labels, scores, n_grid: int = 401) -> tuple[float, dict]:
        labels = np.asarray(labels).astype(np.int64)
        scores = np.asarray(scores, dtype=np.float32)
        if len(scores) == 0:
            return 0.5, metric_dict(labels, scores, 0.5)
        qs = np.linspace(0.0, 1.0, n_grid)
        thresholds = np.unique(np.quantile(scores, qs))
        thresholds = np.unique(np.concatenate([thresholds, np.array([0.5], dtype=np.float32)]))
        best_thr = float(thresholds[0])
        best = None
        for thr in thresholds:
            m = metric_dict(labels, scores, float(thr))
            if best is None or (m["F1"], m["Acc"]) > (best["F1"], best["Acc"]):
                best = m
                best_thr = float(thr)
        return best_thr, best


    seed_everything(SEED)
    codes_path = find_file("codes.jsonl.gz", "codes.jsonl", "codes.jsonl.gz.tmp")
    pairs_path = find_file("pairs.csv.gz", "pairs.csv", "pairs.csv.gz.tmp")
    print("codes:", codes_path, "is_file=", codes_path.is_file())
    print("pairs:", pairs_path, "is_file=", pairs_path.is_file())
    codes = load_codes(codes_path)
    pairs_df = load_pairs(pairs_path)
    train_df = maybe_limit_split(pairs_df, "train", MAX_TRAIN_PAIRS, SEED)
    valid_df = maybe_limit_split(pairs_df, "valid", MAX_VALID_PAIRS, SEED + 1)
    test_df = maybe_limit_split(pairs_df, "test", MAX_TEST_PAIRS, SEED + 2)
    print("pairs:")
    print(pairs_df.groupby(["split", "label"]).size())
    print(f"using train/valid/test={len(train_df):,}/{len(valid_df):,}/{len(test_df):,}")


    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import Dataset, DataLoader

    DEVICE = "cuda" if torch.cuda.is_available() and DEVICE == "cuda" else "cpu"
    print("Device:", DEVICE)

    graphs_path = find_file("graph_spectra.jsonl.gz", "graph_spectra.jsonl", "graph_spectra.jsonl.gz.tmp")
    print("graphs:", graphs_path, "is_file=", graphs_path.is_file())


    def load_graph_records(graph_type: str, needed_ids: set[str]) -> dict[str, dict]:
        out = {}
        with open_text(graphs_path) as f:
            for line in tqdm(f, desc=f"Loading {graph_type.upper()} graphs", unit="code"):
                if not line.strip():
                    continue
                obj = json.loads(line)
                cid = str(obj.get("code_id"))
                if cid not in needed_ids:
                    continue
                layer = obj.get(graph_type, {})
                adj = layer.get("adjacency", {}) if isinstance(layer, dict) else {}
                n = int(adj.get("num_nodes", 0) or 0)
                rows = adj.get("row", []) or []
                cols = adj.get("col", []) or []
                out[cid] = {"n": n, "row": rows, "col": cols}
        return out


    needed_ids = sorted(set(train_df.left_id) | set(train_df.right_id) | set(valid_df.left_id) | set(valid_df.right_id) | set(test_df.left_id) | set(test_df.right_id))
    id_to_row = {cid: i for i, cid in enumerate(needed_ids)}
    records = load_graph_records(GRAPH_TYPE, set(needed_ids))
    print("loaded graphs:", len(records), "/", len(needed_ids))


    def graph_to_arrays(record: dict) -> tuple[np.ndarray, np.ndarray]:
        n0 = int(record.get("n", 0))
        rows = np.asarray(record.get("row", []), dtype=np.int64)
        cols = np.asarray(record.get("col", []), dtype=np.int64)
        keep = (rows < MAX_NODES) & (cols < MAX_NODES)
        rows = rows[keep]
        cols = cols[keep]
        n = min(max(n0, int(rows.max() + 1) if rows.size else 0, int(cols.max() + 1) if cols.size else 0), MAX_NODES)
        a = np.zeros((MAX_NODES, MAX_NODES), dtype=np.float16)
        if rows.size:
            a[rows, cols] = 1.0
            a[cols, rows] = 1.0
        if n > 0:
            a[np.arange(n), np.arange(n)] = 1.0
        deg_out = a.sum(axis=1).astype(np.float32)
        deg_in = a.sum(axis=0).astype(np.float32)
        x = np.zeros((MAX_NODES, NODE_DIM), dtype=np.float16)
        if n > 0:
            x[:n, 0] = 1.0
            x[:n, 1] = np.log1p(deg_out[:n])
            x[:n, 2] = np.log1p(deg_in[:n])
            x[:n, 3] = np.linspace(0.0, 1.0, n, dtype=np.float32)
        denom = np.maximum(a.sum(axis=1, keepdims=True), 1.0)
        a = a / denom
        return a, x


    adj_array = np.zeros((len(needed_ids), MAX_NODES, MAX_NODES), dtype=np.float16)
    node_array = np.zeros((len(needed_ids), MAX_NODES, NODE_DIM), dtype=np.float16)
    missing = 0
    for cid in tqdm(needed_ids, desc="Packing graph tensors"):
        rec = records.get(cid)
        if rec is None:
            missing += 1
            rec = {"n": 0, "row": [], "col": []}
        a, x = graph_to_arrays(rec)
        idx = id_to_row[cid]
        adj_array[idx] = a
        node_array[idx] = x
    print("missing:", missing, "adj:", adj_array.shape, "x:", node_array.shape)


    class PairGraphDataset(Dataset):
        def __init__(self, df: pd.DataFrame):
            self.left = df["left_id"].map(id_to_row).to_numpy(np.int64)
            self.right = df["right_id"].map(id_to_row).to_numpy(np.int64)
            self.y = df["label"].to_numpy(np.float32)

        def __len__(self):
            return len(self.y)

        def __getitem__(self, idx):
            li = self.left[idx]
            ri = self.right[idx]
            return (
                torch.from_numpy(adj_array[li]).float(),
                torch.from_numpy(node_array[li]).float(),
                torch.from_numpy(adj_array[ri]).float(),
                torch.from_numpy(node_array[ri]).float(),
                torch.tensor(self.y[idx], dtype=torch.float32),
            )


    def make_loader(df: pd.DataFrame, shuffle: bool) -> DataLoader:
        return DataLoader(PairGraphDataset(df), batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=NUM_WORKERS, pin_memory=(DEVICE == "cuda"))


    class GMNEncoder(nn.Module):
        def __init__(self):
            super().__init__()
            self.in_proj = nn.Linear(NODE_DIM, HIDDEN_DIM)
            self.msg = nn.Linear(HIDDEN_DIM, HIDDEN_DIM)
            self.match = nn.Linear(HIDDEN_DIM, HIDDEN_DIM)
            self.gru = nn.GRUCell(HIDDEN_DIM * 2, HIDDEN_DIM)
            self.gate = nn.Linear(HIDDEN_DIM, 1)
            self.out_proj = nn.Linear(HIDDEN_DIM, CODE_DIM)

        def readout(self, h, x):
            mask = x[:, :, 0:1]
            gate = torch.sigmoid(self.gate(h)) * mask
            pooled = (gate * h).sum(1) / gate.sum(1).clamp_min(1e-6)
            return self.out_proj(pooled)

        def forward_pair(self, a1, x1, a2, x2):
            h1 = torch.relu(self.in_proj(x1))
            h2 = torch.relu(self.in_proj(x2))
            b, n, d = h1.shape
            mask1 = x1[:, :, 0] > 0
            mask2 = x2[:, :, 0] > 0
            for _ in range(MESSAGE_STEPS):
                m1 = self.msg(torch.bmm(a1, h1))
                m2 = self.msg(torch.bmm(a2, h2))
                sim = torch.bmm(h1, h2.transpose(1, 2)) / math.sqrt(d)
                sim12 = sim.masked_fill(~mask2[:, None, :], -1e4)
                sim21 = sim.transpose(1, 2).masked_fill(~mask1[:, None, :], -1e4)
                att12 = torch.softmax(sim12, dim=2)
                att21 = torch.softmax(sim21, dim=2)
                cross1 = h1 - torch.bmm(att12, h2)
                cross2 = h2 - torch.bmm(att21, h1)
                inp1 = torch.cat([m1, self.match(cross1)], dim=2)
                inp2 = torch.cat([m2, self.match(cross2)], dim=2)
                h1 = self.gru(inp1.reshape(b * n, -1), h1.reshape(b * n, d)).reshape(b, n, d)
                h2 = self.gru(inp2.reshape(b * n, -1), h2.reshape(b, n, d).reshape(b * n, d)).reshape(b, n, d)
            return self.readout(h1, x1), self.readout(h2, x2)


    class PairGraphModel(nn.Module):
        def __init__(self):
            super().__init__()
            self.encoder = GMNEncoder()
            self.classifier = nn.Sequential(
                nn.Linear(CODE_DIM * 4, HIDDEN_DIM * 2),
                nn.ReLU(),
                nn.Dropout(DROPOUT),
                nn.Linear(HIDDEN_DIM * 2, HIDDEN_DIM),
                nn.ReLU(),
                nn.Dropout(DROPOUT),
                nn.Linear(HIDDEN_DIM, 1),
            )

        def forward(self, a1, x1, a2, x2):
            z1, z2 = self.encoder.forward_pair(a1, x1, a2, x2)
            return self.classifier(torch.cat([z1, z2, torch.abs(z1 - z2), z1 * z2], dim=1)).squeeze(1)


    def predict_scores(model, df: pd.DataFrame) -> np.ndarray:
        model.eval()
        scores = []
        loader = make_loader(df, shuffle=False)
        with torch.no_grad():
            for a1, x1, a2, x2, _ in tqdm(loader, desc="Predict", leave=False):
                a1 = a1.to(DEVICE, non_blocking=True)
                x1 = x1.to(DEVICE, non_blocking=True)
                a2 = a2.to(DEVICE, non_blocking=True)
                x2 = x2.to(DEVICE, non_blocking=True)
                logits = model(a1, x1, a2, x2)
                scores.append(torch.sigmoid(logits).detach().cpu().numpy())
        return np.concatenate(scores) if scores else np.array([], dtype=np.float32)


    model = PairGraphModel().to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    criterion = nn.BCEWithLogitsLoss()
    scaler = torch.amp.GradScaler("cuda", enabled=(USE_AMP and DEVICE == "cuda"))
    train_loader = make_loader(train_df, shuffle=True)
    history = []
    best_f1 = -1.0
    best_state = None
    bad_epochs = 0

    for epoch in range(1, EPOCHS + 1):
        model.train()
        total_loss = 0.0
        total_n = 0
        pbar = tqdm(train_loader, desc=f"{METHOD_NAME} epoch {epoch:02d}")
        for a1, x1, a2, x2, y in pbar:
            a1 = a1.to(DEVICE, non_blocking=True)
            x1 = x1.to(DEVICE, non_blocking=True)
            a2 = a2.to(DEVICE, non_blocking=True)
            x2 = x2.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=(USE_AMP and DEVICE == "cuda")):
                logits = model(a1, x1, a2, x2)
                loss = criterion(logits, y)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            total_loss += float(loss.detach().cpu()) * len(y)
            total_n += len(y)
            pbar.set_postfix(loss=total_loss / max(total_n, 1))

        valid_scores = predict_scores(model, valid_df)
        threshold, valid_metrics = choose_threshold(valid_df["label"].to_numpy(), valid_scores)
        row = {
            "Method": METHOD_NAME,
            "epoch": epoch,
            "train_loss": total_loss / max(total_n, 1),
            "valid_P": valid_metrics["P"],
            "valid_R": valid_metrics["R"],
            "valid_F1": valid_metrics["F1"],
            "valid_Acc": valid_metrics["Acc"],
            "threshold": threshold,
        }
        history.append(row)
        print(row)
        if valid_metrics["F1"] > best_f1:
            best_f1 = valid_metrics["F1"]
            best_threshold = threshold
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad_epochs = 0
        else:
            bad_epochs += 1
            if bad_epochs >= PATIENCE:
                print("early stopping")
                break

    pd.DataFrame(history).to_csv(HISTORY_PATH, index=False)
    model.load_state_dict(best_state)
    test_scores = predict_scores(model, test_df)
    test_metrics = metric_dict(test_df["label"].to_numpy(), test_scores, best_threshold)
    result = {
        "Method": METHOD_NAME,
        "BestEpoch": best_epoch,
        "BestValidF1": best_f1,
        **test_metrics,
        "Threshold": best_threshold,
        "TrainPairs": len(train_df),
        "ValidPairs": len(valid_df),
        "TestPairs": len(test_df),
    }
    pd.DataFrame([result]).to_csv(RESULTS_PATH, index=False)
    print(pd.DataFrame([result]))
    print("saved:", RESULTS_PATH)

    if "results_df" in locals():
        return results_df.copy()
    if "result" in locals():
        return pd.DataFrame([result])
    if "row" in locals():
        return pd.DataFrame([row])
    raise RuntimeError("The baseline did not produce a result table.")


from IPython.display import display
import pandas as pd

all_dataset_results = {}
for current_dataset_key in DATASET_KEYS:
    print("\n" + "=" * 96)
    print(f"Running {current_dataset_key.upper()}")
    print("=" * 96)
    dataset_results = run_one_dataset(current_dataset_key)
    dataset_results.insert(0, "Dataset", current_dataset_key.upper())
    all_dataset_results[current_dataset_key] = dataset_results

print("\n" + "=" * 96)
print("Final result tables")
print("=" * 96)
for current_dataset_key in DATASET_KEYS:
    print(f"\n{current_dataset_key.upper()} results")
    display(all_dataset_results[current_dataset_key])

combined_results = pd.concat(
    [all_dataset_results[key] for key in DATASET_KEYS],
    ignore_index=True,
)
combined_path = Path("/kaggle/working") / f"{RUN_LABEL}_combined_dataset_results.csv"
combined_results.to_csv(combined_path, index=False)
print("Combined results:", combined_path)
